# House Prices — K-Fold 교차검증 (LightGBM, OOF) (Colab)

Kaggle [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) 대회를 **K-Fold 교차검증(Cross-Validation)** 으로 학습하는 예제입니다.

- 데이터를 5개 폴드로 나눠, 각 폴드를 검증셋으로 두고 **5개의 모델**을 학습합니다.
- 각 폴드의 검증 예측을 모으면 **OOF(Out-Of-Fold) 예측** 이 되어, 데이터 전체에 대한 신뢰도 높은 검증 점수를 얻습니다.
- 테스트 예측은 **5개 모델 예측의 평균** 으로 만들어 단일 분할보다 안정적입니다.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**하며, 토큰이 **노트북에 내장**되어 있어 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` 가 `/content` 에 생성됩니다.

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "lightgbm", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 House Prices 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비

범주형은 **Ordinal 인코딩**, 숫자형 결측치는 **중앙값**으로 채웁니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
print("train:", train.shape, "| test:", test.shape)

y = train["SalePrice"]
X = train.drop(["Id", "SalePrice"], axis=1)
X_test = test.drop(["Id"], axis=1)

cat_cols = list(X.select_dtypes(include=["object", "string", "category"]).columns)
num_cols = [c for c in X.columns if c not in cat_cols]

for c in cat_cols:
    X[c] = X[c].fillna("NA").astype(str)
    X_test[c] = X_test[c].fillna("NA").astype(str)
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

print("범주형:", len(cat_cols), "| 숫자형:", len(num_cols))

## 3. K-Fold 교차검증 학습

`KFold` 로 데이터를 5개로 나누고, 각 폴드마다 LightGBM 을 학습합니다.
- `oof`: 각 폴드의 검증 예측을 모은 Out-Of-Fold 예측
- `test_pred`: 5개 모델의 테스트 예측 평균

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
import lightgbm as lgb

N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_maes = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = LGBMRegressor(
        n_estimators=2000, learning_rate=0.05, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

    oof[va_idx] = model.predict(X_va)
    test_pred += model.predict(X_test) / N_SPLITS

    fmae = mean_absolute_error(y_va, oof[va_idx])
    fold_maes.append(fmae)
    print(f"Fold {fold} MAE = {fmae:,.1f} | best_iteration {model.best_iteration_}")

## 4. 교차검증 점수

폴드별 점수와 전체 OOF 점수를 확인합니다. OOF MAE 가 리더보드 점수의 좋은 추정치가 됩니다.

In [ ]:
print(f"Mean fold MAE = {np.mean(fold_maes):,.1f}  (+/- {np.std(fold_maes):,.1f})")
print(f"OOF MAE       = {mean_absolute_error(y, oof):,.1f}")

## 5. 제출 파일 생성

In [ ]:
submission_path = os.path.join(WORK_DIR, "submission.csv")
output = pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred})
output.to_csv(submission_path, index=False)
print("Your submission was successfully saved to:", submission_path)
output.head()

## 6. 제출 파일 내려받기

`submission.csv` 가 `/content` 에 생성되었습니다. 좌측 파일창에서 받거나 아래 셀로 바로 내려받으세요.

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[House Prices 제출 페이지](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit)**

- 대회 홈: https://www.kaggle.com/c/house-prices-advanced-regression-techniques
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.